In [4]:
import pandas as pd

df = pd.read_csv("/Users/vishishtareddy/Desktop/ml/data/processed/lifestyle_cleaned.csv")
df.head()

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Stress Level,BMI Category,Heart Rate,Sleep Disorder,Systolic,Diastolic,Activity_Index,Cluster
0,Male,27,Software Engineer,6.1,6,6,Overweight,77,NaN,126,83,176400,2
1,Male,28,Doctor,6.2,6,8,Normal,75,NaN,125,80,600000,2
2,Male,28,Doctor,6.2,6,8,Normal,75,NaN,125,80,600000,2
3,Male,28,Sales Representative,5.9,4,8,Obese,85,Sleep Apnea,140,90,90000,2
4,Male,28,Sales Representative,5.9,4,8,Obese,85,Sleep Apnea,140,90,90000,2


In [5]:
df["BMI_encoded"] = df["BMI Category"].map({
    "Normal": 0,
    "Overweight": 1,
    "Obese": 2
})

In [13]:
features = [
    "Age",
    "Sleep Duration",
    "Stress Level",
    "Heart Rate",
    "Activity_Index",
    "Systolic"
]

X = df[features]
y = df["BMI_encoded"]

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [21]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [22]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9333333333333333

Classification Report:
               precision    recall  f1-score   support

           0       0.90      1.00      0.95        43
           1       1.00      0.83      0.91        30
           2       1.00      1.00      1.00         2

    accuracy                           0.93        75
   macro avg       0.97      0.94      0.95        75
weighted avg       0.94      0.93      0.93        75


Confusion Matrix:
 [[43  0  0]
 [ 5 25  0]
 [ 0  0  2]]


In [23]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))

RF Accuracy: 0.9333333333333333


In [24]:
import pandas as pd

importance = pd.Series(rf.feature_importances_, index=features)
print(importance.sort_values(ascending=False))

Systolic          0.311974
Age               0.267621
Sleep Duration    0.165533
Heart Rate        0.113946
Activity_Index    0.091856
Stress Level      0.049070
dtype: float64


In [25]:
# --- prepare data: drop Age and Systolic, keep same split strategy (stratified) ---
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

# load processed data (adjust path if needed)
df = pd.read_csv("/Users/vishishtareddy/Desktop/ml/data/processed/lifestyle_cleaned.csv")

# encode target if not already encoded
if "BMI_encoded" not in df.columns:
    df["BMI_encoded"] = df["BMI Category"].map({"Normal":0, "Overweight":1, "Obese":2})

# features BEFORE: Age, Sleep Duration, Stress Level, Heart Rate, Activity_Index, Systolic
# new features: remove 'Age' and 'Systolic'
features = ["Sleep Duration", "Stress Level", "Heart Rate", "Activity_Index"]

X = df[features]
y = df["BMI_encoded"]

# stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# scale numeric features (RF doesn't require scaling but keep for consistency with prior work)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# train random forest
rf = RandomForestClassifier(random_state=42, n_estimators=200)
rf.fit(X_train_scaled, y_train)

# evaluate
y_pred = rf.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# feature importance
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("\nFeature importances (sorted):\n", importances)

# optional: display normalized class distribution in test to verify balance
print("\nTest set class distribution (counts):\n", y_test.value_counts())

Accuracy: 0.96

Classification Report:
               precision    recall  f1-score   support

           0       0.93      1.00      0.97        43
           1       1.00      0.93      0.97        30
           2       1.00      0.50      0.67         2

    accuracy                           0.96        75
   macro avg       0.98      0.81      0.87        75
weighted avg       0.96      0.96      0.96        75


Confusion Matrix:
 [[43  0  0]
 [ 2 28  0]
 [ 1  0  1]]

Feature importances (sorted):
 Sleep Duration    0.385029
Activity_Index    0.292273
Heart Rate        0.222962
Stress Level      0.099736
dtype: float64

Test set class distribution (counts):
 BMI_encoded
0    43
1    30
2     2
Name: count, dtype: int64


In [27]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(rf, X, y, cv=skf, scoring="accuracy")

print("CV scores:", scores)
print("Mean accuracy:", np.mean(scores))
print("Std:", np.std(scores))

CV scores: [0.97333333 0.94666667 0.98666667 0.96       0.97297297]
Mean accuracy: 0.967927927927928
Std: 0.013569852528838026
